# IT3091 Bank Marketing · Member 4 (IT24100001)
## Part 3: Export the final pipeline and build the app

Parts 1 and 2 evaluated Member 3's models and fixed the decision rules. Part 3 packages everything into **one file the app can load**, checks it gives exactly the same answers, and writes a ready-to-run app folder.

| Step | What happens | Why it matters |
|---|---|---|
| 0 | Setup | |
| 1 | Folders and pre-flight check | Every file is found before anything is loaded |
| 2 | Write `features.py`, load the inputs | Member 2's rules become importable; Part 2 rules and Member 2/3 files are loaded |
| 3 | Three checks | The exported pipeline must reproduce Member 2, Member 3 and Part 2 exactly |
| 4 | Information the app needs | Dropdown values, defaults, real example customers, test metrics |
| 5 | Assemble the bundle | One object: pipeline + rules + app information |
| 6 | Save `best_pipeline.joblib` and reload it | Proves the saved file works on its own |
| 7 | Write the app folder | `app.py`, `features.py`, pinned `requirements.txt`, guide, demo data |
| 8 | Preview what the app will show | Same function the app calls, on the demo customers |
| 9 | File access log | Only your folder was written to |

**Nothing is trained in this notebook.** It only loads saved, fitted objects and chains them:

`raw customer (15 columns)` → `features.engineer_features` (Member 2 rules) → Member 2's fitted pipeline (`transform` only) → Member 3's tuned model (`predict_proba`) → Part 2 threshold and decile rules

### Files this notebook touches

| Folder (owner) | File | Read / Write | Section |
|---|---|---|---|
| Member 1 (`IT24102060`) | `ds52-train_test_split.joblib` | READ | 2 |
| Member 2 (`IT24101461`) | `IT24101461_selected_data.joblib`, `IT24101461_feature_pipeline.joblib` | READ | 2 |
| Member 3 (`IT24102228`) | the **final model only** (`IT24102228_*_tuned.joblib`, chosen in Part 2) | READ | 2 |
| Member 4 (`IT24100001`) | `IT24100001_part2_threshold_deciles.joblib` (from your Part 2) | READ | 2 |
| Member 4 (`IT24100001/app`) | `features.py` | WRITE | 2 |
| Member 4 (`IT24100001/app`) | `best_pipeline.joblib` | WRITE | 6 |
| Member 4 (`IT24100001/app`) | `app.py`, `requirements.txt`, `README.md`, 2 demo CSVs | WRITE | 7 |
| Member 4 (`IT24100001`) | `IT24100001_part3_file_access_log.csv` | WRITE | 9 |

Nothing is ever written to the Member 1, 2 or 3 folders. Every read and write prints a `[READ ]` or `[WRITE]` line.

## 0. Setup

In [1]:
import os, sys, platform, importlib, warnings, subprocess
warnings.filterwarnings("ignore")

def ensure(pkg):
    try:
        return __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        return __import__(pkg)

ensure("lightgbm"); ensure("xgboost")   # needed to load Member 3's boosted models

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

RANDOM_STATE = 42
pd.set_option("display.width", 160)
print("Python      :", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Python      : 3.13.16
scikit-learn: 1.6.1


## 1. Folders and pre-flight check
**Edit the folder names below if your files are somewhere else.** Each file is looked for first in the folder of the member who made it; if it is not there, the other folders are checked and a `NOTE` line says where it was found.

All app files go into a new subfolder, **`IT24100001/app`**. That is the folder you download to your laptop.

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # running outside Colab

# The project lives in YOUR My Drive:
#   My Drive / ML & DWBI Assignment / ML Assignment / IT24102060, IT24101461, IT24102228, IT24100001
PROJECT_DIR = os.environ.get("M4_PROJECT_DIR",
                             "/content/drive/MyDrive/ML & DWBI Assignment/ML Assignment")
assert os.path.isdir(PROJECT_DIR), (
    f"Folder not found: {PROJECT_DIR}\n"
    "Check the folder exists in My Drive with exactly this name, or change PROJECT_DIR above.")
print("Project folder:", PROJECT_DIR)

# ---- one folder per member: CHANGE HERE if needed -----------------------------------
FOLDERS = {
    "Member 1": os.path.join(PROJECT_DIR, "IT24102060"),   # train/test split (READ only)
    "Member 2": os.path.join(PROJECT_DIR, "IT24101461"),   # feature engineering (READ only)
    "Member 3": os.path.join(PROJECT_DIR, "IT24102228"),   # tuned models (READ only)
    "Member 4": os.path.join(PROJECT_DIR, "IT24100001"),   # YOU: Part 1 and 2 outputs
}
MEMBER_4_DIR = FOLDERS["Member 4"]
APP_DIR = os.path.join(MEMBER_4_DIR, "app")                # everything the app needs
os.makedirs(APP_DIR, exist_ok=True)

ACCESS_LOG = []

def short(p):
    return os.path.relpath(p, PROJECT_DIR)

def locate(owner, filename):
    expected = os.path.join(FOLDERS[owner], filename)
    if os.path.exists(expected):
        return expected, owner
    for other, folder in FOLDERS.items():
        candidate = os.path.join(folder, filename)
        if other != owner and os.path.exists(candidate):
            return candidate, other
    raise FileNotFoundError(f"{filename} ({owner}'s file) not found in any member folder.")

def read_file(owner, filename, why):
    full, found_in = locate(owner, filename)
    if found_in != owner:
        print(f"NOTE  {filename} is not in the {owner} folder; using the copy in the {found_in} folder")
    print(f"[READ ] {found_in} folder -> {short(full)}   ({why})")
    ACCESS_LOG.append({"action": "READ", "file owner": owner, "folder used": found_in,
                       "path": short(full), "purpose": why})
    return joblib.load(full)

def log_write(full, why):
    print(f"[WRITE] Member 4 folder -> {short(full)}   ({why})")
    ACCESS_LOG.append({"action": "WRITE", "file owner": "Member 4", "folder used": "Member 4",
                       "path": short(full), "purpose": why})

def write_text(name, text, why):
    full = os.path.join(APP_DIR, name)
    with open(full, "w", encoding="utf-8") as fh:
        fh.write(text)
    log_write(full, why)
    return full

for owner, folder in FOLDERS.items():
    print(f"{owner}: {folder}  {'(exists)' if os.path.isdir(folder) else '(FOLDER NOT FOUND)'}")
print("App folder:", APP_DIR)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment
Member 1: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24102060  (exists)
Member 2: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24101461  (exists)
Member 3: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24102228  (exists)
Member 4: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24100001  (exists)
App folder: /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24100001/app


In [3]:
# Part 2 decides which Member 3 model is final, so check for its output first
REQUIRED = [("Member 4", "IT24100001_part2_threshold_deciles.joblib"),
            ("Member 1", "ds52-train_test_split.joblib"),
            ("Member 2", "IT24101461_selected_data.joblib"),
            ("Member 2", "IT24101461_feature_pipeline.joblib")]
rows = []
for owner, f in REQUIRED:
    try:
        _, found_in = locate(owner, f)
        rows.append({"made by": owner, "file": f, "found in": f"{found_in} folder",
                     "status": "OK" if found_in == owner else "OK (copy)"})
    except FileNotFoundError:
        rows.append({"made by": owner, "file": f, "found in": "-", "status": "MISSING"})
preflight = pd.DataFrame(rows)
assert not (preflight["status"] == "MISSING").any(), "A required file is missing. Run Part 2 first."
preflight

,made by,file,found in,status
0,Member 4,IT24100001_part2_threshold_deciles.joblib,Member 4 folder,OK
1,Member 1,ds52-train_test_split.joblib,Member 1 folder,OK
2,Member 2,IT24101461_selected_data.joblib,Member 2 folder,OK
3,Member 2,IT24101461_feature_pipeline.joblib,Member 2 folder,OK


## 2. Write `features.py` and load the inputs

### 2.1 `features.py`
The saved pipeline's first step is `features.engineer_features`: Member 2's 8 feature rules (sections 1.3 to 1.9 of their notebook), written as one function. Python saves a **reference** to that function, not its code, so `features.py` must sit next to `app.py` wherever the app runs. It also contains the helpers the app uses to clean input, place customers into deciles, and score them.

**File access:** WRITE `features.py` to the `app` folder.

In [4]:
FEATURES_PY = r'''"""
features.py  (Member 4, IT24100001)

Shared by the Part 3 notebook and app.py. Keep this file in the same folder as
app.py: the saved pipeline (best_pipeline.joblib) refers to
features.engineer_features by name, so Python must be able to import it.

  engineer_features(df)   Member 2's feature rules, copied exactly (sections 1.3 to 1.9)
  prepare_input(df, b)    cleans uploaded or typed customer data before scoring
  assign_decile(s, edges) places scores into priority deciles using the saved cut points
  score_customers(df, b)  score -> CALL decision -> decile -> expected rate, ranked
"""

import numpy as np
import pandas as pd

RAW_COLUMNS = ["age", "job", "marital", "education", "default", "balance", "housing",
               "loan", "contact", "day", "month", "campaign", "pdays", "previous", "poutcome"]
NUMERIC_COLUMNS = ["age", "balance", "day", "campaign", "pdays", "previous"]
NOT_FEATURES = ["duration", "y"]   # duration is only known after the call (leakage)


def engineer_features(df):
    """Member 2's 8 engineered features. Fixed rules only: nothing is learned from data."""
    out = df.copy()
    out = out.drop(columns=[c for c in NOT_FEATURES if c in out.columns])

    # 1.3  pdays == -1 means the customer was never contacted before
    out["previously_contacted"] = (out["pdays"] != -1).astype(int)
    # 1.4  -1 replaced by 0; previously_contacted keeps the "never" information
    out["pdays_clean"] = out["pdays"].replace(-1, 0)
    # 1.5  loans and credit default
    out["loan_count"] = (out["housing"] == "yes").astype(int) + (out["loan"] == "yes").astype(int)
    out["has_credit_default"] = (out["default"] == "yes").astype(int)
    # 1.6  previous campaign success
    out["previous_campaign_success"] = (out["poutcome"] == "success").astype(int)
    # 1.7  age group
    out["age_group"] = pd.cut(out["age"], bins=[17, 30, 40, 50, 60, np.inf],
                              labels=["18-30", "31-40", "41-50", "51-60", "60+"])
    # 1.8  balance group
    out["balance_group"] = pd.cut(out["balance"], bins=[-np.inf, -1, 0, 1000, 5000, np.inf],
                                  labels=["negative", "zero", "low", "medium", "high"])
    # 1.9  campaign contact group
    out["campaign_contact_group"] = pd.cut(out["campaign"], bins=[0, 1, 2, 4, np.inf],
                                           labels=["first_contact", "second_contact",
                                                   "three_to_four", "five_or_more"])
    return out


def prepare_input(df, bundle=None):
    """Clean customer rows so they match the training format.

    Returns (clean_df, warnings). Raises ValueError if required columns are missing.
    Extra columns (for example a customer_id) are kept and passed through untouched.
    """
    df = df.copy()
    df.columns = [str(c).strip().strip('"').lower() for c in df.columns]
    missing = [c for c in RAW_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    warnings = []
    for c in NUMERIC_COLUMNS:
        before = df[c].isna().sum()
        df[c] = pd.to_numeric(df[c], errors="coerce")
        bad = int(df[c].isna().sum() - before)
        if bad:
            warnings.append(f"{bad} row(s) had a non-numeric '{c}'; the training median is used instead.")
    for c in [c for c in RAW_COLUMNS if c not in NUMERIC_COLUMNS]:
        df[c] = df[c].astype(str).str.strip().str.strip('"').str.lower()
        if bundle is not None and c in bundle.get("categories", {}):
            unknown = sorted(set(df[c]) - set(bundle["categories"][c]))
            if unknown:
                warnings.append(f"'{c}' has values never seen in training {unknown}; "
                                f"they are treated as 'no information'.")
    return df, warnings


def assign_decile(scores, edges):
    """Decile 1 = highest scores, 10 = lowest. `edges` are the 9 saved cut points (ascending)."""
    return 10 - np.searchsorted(np.asarray(edges), np.asarray(scores), side="right")


def score_customers(df, bundle):
    """Score raw customers and return them as a ranked calling queue.

    Adds: rank, score, recommendation, priority_decile, expected_rate.
    Uses only saved objects: nothing is fitted here.
    """
    clean, _ = prepare_input(df, bundle)
    score = bundle["pipeline"].predict_proba(clean[RAW_COLUMNS])[:, 1]   # Member 3 model score
    decile = assign_decile(score, bundle["decile_edges"])                 # Part 2 cut points
    rates = bundle["decile_expected_rate"]                                # Part 2 expected rates

    out = clean.copy()
    out["score"] = score
    out["recommendation"] = np.where(score >= bundle["threshold"], "CALL", "Lower-cost channel")
    out["priority_decile"] = decile
    out["expected_rate"] = [rates[int(d)] for d in decile]
    out = out.sort_values("score", ascending=False).reset_index(drop=True)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    return out


def decile_ranges(edges):
    """Readable score range for each decile, for display."""
    e = list(edges)
    rows = []
    for d in range(1, 11):
        if d == 1:
            rng = f"score >= {e[8]:.3f}"
        elif d == 10:
            rng = f"score < {e[0]:.3f}"
        else:
            rng = f"{e[9 - d]:.3f} <= score < {e[10 - d]:.3f}"
        rows.append({"decile": d, "score range": rng})
    return pd.DataFrame(rows)
'''

write_text("features.py", FEATURES_PY, "Member 2 feature rules + scoring helpers")
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
sys.dont_write_bytecode = True   # keep the app folder clean (no __pycache__)
import features
importlib.reload(features)   # pick up the file just written
print("features.py imported from", features.__file__)

[WRITE] Member 4 folder -> IT24100001/app/features.py   (Member 2 feature rules + scoring helpers)
features.py imported from /content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24100001/app/features.py


### 2.2 Load the inputs
**File access:** READ the Part 2 output from the **Member 4** folder, the **Member 1** and **Member 2** files, and **only the final model** from the **Member 3** folder.

In [5]:
# >>> MEMBER 4 FOLDER: Part 2 decided the final model, threshold and decile rules
part2 = read_file("Member 4", "IT24100001_part2_threshold_deciles.joblib", "Part 2 decision rules")
FINAL_MODEL = part2["final_model"]
THRESHOLD = float(part2["threshold"])
DECILE_EDGES = np.asarray(part2["decile_edges"], dtype=float)
DECILE_EXPECTED_RATE = {int(k): float(v) for k, v in part2["decile_expected_rate"].items()}
print(f"\nFinal model: {FINAL_MODEL} ({part2['chosen_by']}) | threshold {THRESHOLD:.4f}\n")

# >>> MEMBER 1 FOLDER: raw customers (the app's input format) and labels
X_train_raw, X_test_raw, y_train_raw, y_test_raw = read_file(
    "Member 1", "ds52-train_test_split.joblib", "raw columns, categories, demo customers")

# >>> MEMBER 2 FOLDER: processed test data (for the checks) and the fitted feature pipeline
_, X_test, _, y_test_s, selected_names = read_file(
    "Member 2", "IT24101461_selected_data.joblib", "processed test set for the checks")
feature_pipeline = read_file(
    "Member 2", "IT24101461_feature_pipeline.joblib", "fitted scaler + encoder + selector")

# >>> MEMBER 3 FOLDER: only the model Part 2 chose
final_model = read_file("Member 3", part2["final_model_file"], f"final model: {FINAL_MODEL}")

def to01(y):
    y = pd.Series(y)
    return (y if pd.api.types.is_numeric_dtype(y) else y.map({"no": 0, "yes": 1})).astype(int).to_numpy()
y_test = to01(y_test_s)
assert np.array_equal(to01(y_test_raw), y_test), "Member 1 and Member 2 test labels differ."
assert len(X_test_raw) == len(X_test), "Member 1 and Member 2 test sets have different sizes."
print("\nPASS Member 1 and Member 2 test sets line up")

[READ ] Member 4 folder -> IT24100001/IT24100001_part2_threshold_deciles.joblib   (Part 2 decision rules)

Final model: XGBoost (highest OOF F1 at tuned threshold) | threshold 0.2033

[READ ] Member 1 folder -> IT24102060/ds52-train_test_split.joblib   (raw columns, categories, demo customers)
[READ ] Member 2 folder -> IT24101461/IT24101461_selected_data.joblib   (processed test set for the checks)
[READ ] Member 2 folder -> IT24101461/IT24101461_feature_pipeline.joblib   (fitted scaler + encoder + selector)
[READ ] Member 3 folder -> IT24102228/IT24102228_xgboost_tuned.joblib   (final model: XGBoost)

PASS Member 1 and Member 2 test sets line up


## 3. Three checks before anything is exported

| Check | Question | Why |
|---|---|---|
| A | Does `features.py` + Member 2's pipeline reproduce Member 2's 30 processed test columns exactly? | Proves the copied feature rules are identical |
| B | Does the chained pipeline on **raw** test rows give exactly the same scores as Member 3's model on processed rows? | Proves the app predicts with Member 3's model, unchanged |
| C | Do those scores give the same decile counts and CALL decisions as Part 2? | Proves the app applies Part 2's rules correctly |

If any check fails, the export in section 6 does not run.

In [6]:
# Compatibility fix Member 3 also needed: the pipeline was saved with a different scikit-learn version
def repair(obj):
    if isinstance(obj, ColumnTransformer) and getattr(obj, "force_int_remainder_cols", None) == "deprecated":
        obj.force_int_remainder_cols = False
    for _, child in getattr(obj, "steps", []):
        repair(child)
    for t in getattr(obj, "transformers", []):
        if t[1] not in ("drop", "passthrough"):
            repair(t[1])
repair(feature_pipeline)

def classifier_of(model):
    # Member 3 saved each model as Pipeline([("classifier", ...)]): take the fitted classifier itself
    return model.named_steps["classifier"] if hasattr(model, "named_steps") else model

checks = {}

# A. copied feature rules + Member 2's fitted pipeline == Member 2's saved processed data
reproduced = feature_pipeline.transform(features.engineer_features(X_test_raw))
checks["A. features.py + Member 2 pipeline reproduce the 30 processed columns"] = (
    reproduced.shape == np.asarray(X_test).shape and np.allclose(reproduced, X_test))

# The final pipeline: nothing is fitted here, every step is already fitted
best_pipeline = Pipeline([
    ("engineer", FunctionTransformer(features.engineer_features)),   # Member 2 rules
    ("features", feature_pipeline),                                  # Member 2 fitted pipeline
    ("model", classifier_of(final_model)),                           # Member 3 fitted model
])

# B. chained pipeline on raw rows == Member 3's model on processed rows
scores_member3 = final_model.predict_proba(X_test)[:, 1]
scores_chain = best_pipeline.predict_proba(X_test_raw)[:, 1]
checks["B. chained pipeline == Member 3's model (max difference "
       f"{np.max(np.abs(scores_chain - scores_member3)):.1e})"] = np.allclose(scores_chain, scores_member3)

# C. same decile counts and CALL decisions as Part 2
deciles_now = features.assign_decile(scores_chain, DECILE_EDGES)
counts_now = pd.Series(deciles_now).value_counts().reindex(range(1, 11), fill_value=0).to_numpy()
counts_part2 = part2["decile_table"]["customers"].reindex(range(1, 11), fill_value=0).to_numpy()
calls_now = int((scores_chain >= THRESHOLD).sum())
calls_part2 = int(part2["test_threshold_table"].loc[(FINAL_MODEL, "tuned"), "Customers called"])
checks["C. decile counts and CALL decisions match Part 2"] = (
    np.array_equal(counts_now, counts_part2) and calls_now == calls_part2)

for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
EXPORT_OK = all(checks.values())
print("\nReady to export." if EXPORT_OK else "\nNOT exported: fix the failed check first.")

PASS A. features.py + Member 2 pipeline reproduce the 30 processed columns
PASS B. chained pipeline == Member 3's model (max difference 0.0e+00)
PASS C. decile counts and CALL decisions match Part 2

Ready to export.


## 4. Information the app needs

| Item | Used for |
|---|---|
| `categories` | Dropdown choices, and warnings for unknown values in uploads (from training data) |
| `defaults` | The "typical customer" the form starts with (training median / most common value) |
| `examples` | Three **real test customers** to load into the form: strongest, weakest, middle |
| `test_metrics` | The sidebar numbers (test set, tuned threshold) |
| `top_drivers` | The "Why this result?" panel: which features the final model relies on most (permutation importance on the test set, computed here) |

The examples come from the test set, which no model was trained on, so they count as genuinely new customers.

In [7]:
RAW = features.RAW_COLUMNS
is_num = pd.api.types.is_numeric_dtype

def plain(row):
    # numpy types -> plain Python, so Streamlit inputs accept them
    return {c: (int(row[c]) if is_num(X_train_raw[c]) else str(row[c])) for c in RAW}

categories = {c: sorted(X_train_raw[c].astype(str).unique().tolist()) for c in RAW if not is_num(X_train_raw[c])}
defaults = {c: (int(round(X_train_raw[c].median())) if is_num(X_train_raw[c]) else str(X_train_raw[c].mode()[0]))
            for c in RAW}

test_raw = X_test_raw.reset_index(drop=True)
order = np.argsort(scores_chain)
examples = {
    "Real test customer: strongest prospect": plain(test_raw.iloc[order[-1]]),
    "Real test customer: middle of the pack": plain(test_raw.iloc[order[len(order) // 2]]),
    "Real test customer: weakest prospect": plain(test_raw.iloc[order[0]]),
}

pred = (scores_chain >= THRESHOLD).astype(int)
test_metrics = {"Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred),
                "F1": f1_score(y_test, pred), "F1 at 0.5": f1_score(y_test, (scores_chain >= 0.5).astype(int)),
                "ROC-AUC": roc_auc_score(y_test, scores_chain),
                "PR-AUC": average_precision_score(y_test, scores_chain)}

# Which original features the final model relies on (permutation importance, test set).
# Shuffle one column at a time; a bigger drop in PR-AUC means the model relies on it more.
from sklearn.inspection import permutation_importance
CATEGORICAL = ["job", "marital", "education", "default", "housing", "loan", "contact", "month",
               "poutcome", "age_group", "balance_group", "campaign_contact_group"]   # Member 2
def original_feature(col):
    prefix, base = col.split("__", 1) if "__" in col else ("", col)
    if prefix == "numeric":
        return base
    matches = [c for c in CATEGORICAL if base.startswith(c + "_")]
    return max(matches, key=len) if matches else base

perm = permutation_importance(final_model, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
top_drivers = (pd.Series(perm.importances_mean, index=[original_feature(str(n)) for n in selected_names])
               .groupby(level=0).sum().sort_values(ascending=False).head(8).round(4).astype(float).to_dict())
print("Top drivers:", top_drivers)

print("Defaults:", defaults)
print("\nTest metrics at the tuned threshold:", {k: round(v, 4) for k, v in test_metrics.items()})
for name, ex in examples.items():
    s = best_pipeline.predict_proba(pd.DataFrame([ex]))[:, 1][0]
    print(f"\n{name}: score {s:.3f}, decile {features.assign_decile([s], DECILE_EDGES)[0]}")
    print("  ", ex)

Top drivers: {'previous_campaign_success': 0.1014, 'month': 0.0548, 'pdays_clean': 0.0407, 'contact': 0.0318, 'day': 0.0312, 'age': 0.016, 'loan_count': 0.0117, 'balance': 0.0082}
Defaults: {'age': 39, 'job': 'blue-collar', 'marital': 'married', 'education': 'secondary', 'default': 'no', 'balance': 451, 'housing': 'yes', 'loan': 'no', 'contact': 'cellular', 'day': 16, 'month': 'may', 'campaign': 2, 'pdays': -1, 'previous': 0, 'poutcome': 'unknown'}

Test metrics at the tuned threshold: {'Precision': 0.4504, 'Recall': 0.4802, 'F1': 0.4648, 'F1 at 0.5': 0.3289, 'ROC-AUC': np.float64(0.7885), 'PR-AUC': np.float64(0.4319)}

Real test customer: strongest prospect: score 0.958, decile 1
   {'age': 41, 'job': 'unemployed', 'marital': 'married', 'education': 'tertiary', 'default': 'no', 'balance': 0, 'housing': 'no', 'loan': 'no', 'contact': 'cellular', 'day': 29, 'month': 'apr', 'campaign': 1, 'pdays': 181, 'previous': 3, 'poutcome': 'success'}

Real test customer: middle of the pack: score 0

## 5. Assemble the bundle
Everything the app needs, in one dictionary. The `pipeline` inside it is the chain from section 3, built from Member 2's and Member 3's **original fitted objects**.

In [8]:
import lightgbm, xgboost
versions = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
            "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__}

bundle = {
    "pipeline": best_pipeline,                       # Member 2 rules + Member 2 pipeline + Member 3 model
    "model_name": FINAL_MODEL,
    "model_file": part2["final_model_file"],
    "threshold": THRESHOLD,                          # Part 2
    "decile_edges": DECILE_EDGES,                    # Part 2
    "decile_expected_rate": DECILE_EXPECTED_RATE,    # Part 2
    "decile_table": part2["decile_table"],           # Part 2, test-set check
    "base_rate": float(part2["base_rate"]),
    "raw_columns": RAW,
    "model_columns": [str(n) for n in selected_names],
    "categories": categories,
    "defaults": defaults,
    "examples": examples,
    "test_metrics": test_metrics,
    "top_drivers": top_drivers,
    "versions": versions,
    "created_by": "IT24100001 (Member 4), Part 3",
}
print("Bundle keys:", list(bundle))
print("Versions   :", versions)

Bundle keys: ['pipeline', 'model_name', 'model_file', 'threshold', 'decile_edges', 'decile_expected_rate', 'decile_table', 'base_rate', 'raw_columns', 'model_columns', 'categories', 'defaults', 'examples', 'test_metrics', 'top_drivers', 'versions', 'created_by']
Versions   : {'python': '3.13.16', 'scikit-learn': '1.6.1', 'numpy': '2.1.3', 'pandas': '2.2.3', 'joblib': '1.6.0', 'xgboost': '3.4.1', 'lightgbm': '4.6.0'}


## 6. Save `best_pipeline.joblib` and reload it
The reload test opens the saved file from disk, exactly as the app will, and scores the test set again.

**File access:** WRITE `best_pipeline.joblib` to the `app` folder, then read it back.

In [9]:
BUNDLE_FILE = os.path.join(APP_DIR, "best_pipeline.joblib")
assert EXPORT_OK, "Checks in section 3 failed, so nothing is saved."

# >>> MEMBER 4 FOLDER: the one file the app loads
joblib.dump(bundle, BUNDLE_FILE, compress=3)
log_write(BUNDLE_FILE, "pipeline + Part 2 rules + app information")
print(f"Size: {os.path.getsize(BUNDLE_FILE) / 1e6:.1f} MB")

# Reload test: what the app does
reloaded = joblib.load(BUNDLE_FILE)
ACCESS_LOG.append({"action": "READ", "file owner": "Member 4", "folder used": "Member 4",
                   "path": short(BUNDLE_FILE), "purpose": "reload test"})
again = reloaded["pipeline"].predict_proba(X_test_raw)[:, 1]
print("PASS" if np.allclose(again, scores_member3) else "FAIL",
      "the saved file reproduces Member 3's test scores")

[WRITE] Member 4 folder -> IT24100001/app/best_pipeline.joblib   (pipeline + Part 2 rules + app information)
Size: 0.2 MB
PASS the saved file reproduces Member 3's test scores


## 7. Write the app folder

| File | Contents |
|---|---|
| `app.py` | The Streamlit app (3 tabs) |
| `requirements.txt` | **Exact** library versions from this Colab session, so the saved pipeline loads on your laptop |
| `README.md` | Step-by-step guide: laptop setup, running, demo script, troubleshooting |
| `demo_new_customers.csv` | 200 random **test** customers with the outcome removed, plus a `customer_id` |
| `demo_answer_key.csv` | What those 200 customers really did, revealed at the end of the demo |

Only the library the final model needs is listed (`lightgbm` or `xgboost`), to keep installation simple.

In [10]:
APP_PY = r'''"""
Term Deposit Call Planner  (Member 4, IT24100001)
Dual-lens decision interface for the Bank Marketing project.

  Tab 1  Single customer   (ML lens)          score one customer and explain the decision
  Tab 2  Campaign queue    (Analytical lens)  rank a CSV of customers into a calling queue
  Tab 3  How it works                         the rules behind every decision

Run from this folder:   streamlit run app.py
Needs in the same folder: best_pipeline.joblib, features.py
Optional (for the demo): demo_new_customers.csv, demo_answer_key.csv

Nothing is trained in this app. It only loads the saved pipeline
(Member 2 feature steps + Member 3 model) and the Part 2 decision rules.
"""
import os

import joblib
import numpy as np
import pandas as pd
import sklearn
import streamlit as st

import features  # must sit next to app.py: the saved pipeline refers to it by name

HERE = os.path.dirname(os.path.abspath(__file__))
BUNDLE_PATH = os.path.join(HERE, "best_pipeline.joblib")
DEMO_CUSTOMERS = os.path.join(HERE, "demo_new_customers.csv")
DEMO_ANSWERS = os.path.join(HERE, "demo_answer_key.csv")

MONTHS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
FIELDS = {   # label and help text for every raw input column
    "age": ("Age", "Customer age in years"),
    "job": ("Job", "Type of job"),
    "marital": ("Marital status", None),
    "education": ("Education", None),
    "default": ("Has credit in default?", None),
    "balance": ("Average yearly balance (EUR)", "Can be negative"),
    "housing": ("Has housing loan?", None),
    "loan": ("Has personal loan?", None),
    "contact": ("Contact type", "How the customer is contacted"),
    "day": ("Last contact day of month", "1 to 31"),
    "month": ("Last contact month", None),
    "campaign": ("Contacts in this campaign", "Including the last contact"),
    "pdays": ("Days since last contact in a previous campaign", "-1 means never contacted before"),
    "previous": ("Contacts before this campaign", None),
    "poutcome": ("Outcome of the previous campaign", "'unknown' if never contacted"),
}
GROUPS = [("Customer profile", ["age", "job", "marital", "education"]),
          ("Finances", ["default", "balance", "housing", "loan"]),
          ("This campaign", ["contact", "day", "month", "campaign"]),
          ("Previous campaigns", ["pdays", "previous", "poutcome"])]

st.set_page_config(page_title="Term Deposit Call Planner", layout="wide")


# ------------------------------------------------------------------ load the saved bundle
@st.cache_resource
def load_bundle(path):
    return joblib.load(path)


if not os.path.exists(BUNDLE_PATH):
    st.error("best_pipeline.joblib was not found next to app.py. Run the Part 3 notebook, "
             "download the app folder, and start the app from inside that folder.")
    st.stop()

bundle = load_bundle(BUNDLE_PATH)
THRESHOLD = bundle["threshold"]
BASE_RATE = bundle["base_rate"]
RATES = bundle["decile_expected_rate"]


def as_percent(df, cols):
    # display copy only: 0.286 -> "28.6%"
    out = df.copy()
    for c in cols:
        out[c] = out[c].map(lambda v: f"{v:.1%}" if pd.notna(v) else "")
    return out


def options_for(col):
    opts = list(bundle["categories"][col])
    if col == "month":
        opts = [m for m in MONTHS if m in opts] + [m for m in opts if m not in MONTHS]
    return opts


# ------------------------------------------------------------------ sidebar
with st.sidebar:
    st.header("Model in use")
    st.write(f"**{bundle['model_name']}** (Member 3, tuned)")
    m = bundle["test_metrics"]
    c1, c2 = st.columns(2)
    c1.metric("Test F1", f"{m['F1']:.3f}")
    c2.metric("Test PR-AUC", f"{m['PR-AUC']:.3f}")
    c1.metric("Precision", f"{m['Precision']:.3f}")
    c2.metric("Recall", f"{m['Recall']:.3f}")
    st.caption(f"Decision threshold: **{THRESHOLD:.3f}** (tuned in Part 2; default would be 0.5).")
    st.caption(f"Average subscription rate: **{BASE_RATE:.1%}**.")
    st.caption("`duration` is never used: it is only known after the call.")
    trained_with = bundle.get("versions", {}).get("scikit-learn")
    if trained_with and trained_with != sklearn.__version__:
        st.warning(f"scikit-learn {sklearn.__version__} is installed but the pipeline was saved "
                   f"with {trained_with}. Install the version in requirements.txt.")

st.title("Term Deposit Call Planner")
st.caption("Who should the call centre phone first? Scores come from Member 3's tuned model; "
           "the call rule and priority groups come from Part 2.")
tab1, tab2, tab3 = st.tabs(["Single customer (ML lens)",
                            "Campaign queue (Analytical lens)",
                            "How it works"])

# ------------------------------------------------------------------ tab 1: single customer
with tab1:
    examples = bundle.get("examples", {})
    for c, v in bundle["defaults"].items():            # first run: start from a typical customer
        st.session_state.setdefault(f"in_{c}", v)

    def load_example():
        choice = st.session_state["example_choice"]
        row = bundle["defaults"] if choice == "Typical customer (training averages)" else examples[choice]
        for c, v in row.items():
            st.session_state[f"in_{c}"] = v

    st.selectbox("Start from", ["Typical customer (training averages)"] + list(examples),
                 key="example_choice", on_change=load_example,
                 help="The examples are real customers from the test set, which no model was trained on.")

    with st.form("customer_form"):
        cols = st.columns(4)
        for col, (title, fields) in zip(cols, GROUPS):
            with col:
                st.markdown(f"**{title}**")
                for f in fields:
                    label, help_text = FIELDS[f]
                    if f in bundle["categories"]:
                        st.selectbox(label, options_for(f), key=f"in_{f}", help=help_text)
                    else:
                        st.number_input(label, step=1, key=f"in_{f}", help=help_text)
        st.form_submit_button("Score this customer", type="primary")

    customer = pd.DataFrame([{c: st.session_state[f"in_{c}"] for c in features.RAW_COLUMNS}])
    result = features.score_customers(customer, bundle).iloc[0]
    d = int(result["priority_decile"])
    rate = result["expected_rate"]

    st.divider()
    r1, r2, r3 = st.columns(3)
    r1.metric("Recommendation", result["recommendation"],
              "score is above the call threshold" if result["recommendation"] == "CALL"
              else "score is below the call threshold", delta_color="off")
    r2.metric("Priority decile", f"{d} of 10", "1 = call first", delta_color="off")
    r3.metric("Expected subscription rate", f"{rate:.0%}",
              f"{rate / BASE_RATE:.1f}x the average customer", delta_color="off")
    st.caption(f"Model score {result['score']:.3f} against threshold {THRESHOLD:.3f}. "
               f"The expected rate is the share of training customers in decile {d} who really "
               f"subscribed; it is shown instead of the raw score, which class weighting inflates.")

    with st.expander("Why this result?"):
        if bundle["model_name"] == "Logistic Regression":
            clf = bundle["pipeline"].named_steps["model"]
            clean, _ = features.prepare_input(customer, bundle)
            x = np.asarray(bundle["pipeline"][:-1].transform(clean[features.RAW_COLUMNS]))[0]
            contrib = (pd.DataFrame({"feature": bundle["model_columns"], "effect": x * clf.coef_[0]})
                       .assign(size=lambda t: t.effect.abs())
                       .sort_values("size", ascending=False).head(8))
            contrib["direction"] = np.where(contrib.effect > 0, "raises the score", "lowers the score")
            st.write("Biggest influences on **this customer's** score:")
            st.dataframe(contrib[["feature", "direction", "effect"]].round(3), hide_index=True)
        elif bundle.get("top_drivers"):
            st.write(f"Features the {bundle['model_name']} relies on most "
                     "(permutation importance on the test set):")
            st.dataframe(pd.Series(bundle["top_drivers"], name="drop in PR-AUC when shuffled")
                         .rename_axis("feature").reset_index().round(4), hide_index=True)
        else:
            st.write("See the explainability section of the report (Part 1).")
        st.write(f"Decile {d} covers {features.decile_ranges(bundle['decile_edges']).loc[d - 1, 'score range']}.")

# ------------------------------------------------------------------ tab 2: campaign queue
with tab2:
    st.subheader("Build a calling queue")
    st.caption("Upload customers in the bank-full.csv format (comma or semicolon separated). "
               "Extra columns such as a customer ID are kept. `y` and `duration` are ignored.")

    left, right = st.columns([3, 1])
    upload = left.file_uploader("Customer file (CSV)", type=["csv"])
    if os.path.exists(DEMO_CUSTOMERS) and right.button("Use the demo customers"):
        st.session_state["use_demo"] = True
    if upload is not None:
        st.session_state["use_demo"] = False

    raw = None
    if upload is not None:
        raw = pd.read_csv(upload, sep=None, engine="python")
        source = upload.name
    elif st.session_state.get("use_demo"):
        raw = pd.read_csv(DEMO_CUSTOMERS)
        source = "demo_new_customers.csv (test-set customers, never seen in training)"

    queue = None
    if raw is None:
        st.info("Upload a CSV, or press **Use the demo customers**.")
    else:
        try:
            _, notes = features.prepare_input(raw, bundle)
            queue = features.score_customers(raw, bundle)
        except ValueError as err:
            st.error(str(err))
    if queue is not None:
        for n in notes:
            st.warning(n)
        st.success(f"Scored {len(queue):,} customers from {source}.")

        n_calls = st.slider("How many calls can the team make?", 1, len(queue),
                            max(1, int(round(len(queue) * 0.2))))
        top = queue.head(n_calls)
        expected = top["expected_rate"].sum()
        k1, k2, k3, k4 = st.columns(4)
        k1.metric("Customers", f"{len(queue):,}")
        k2.metric("Marked CALL by the model", f"{(queue['recommendation'] == 'CALL').sum():,}")
        k3.metric(f"Expected subscribers in top {n_calls}", f"{expected:,.0f}",
                  f"random calling: {n_calls * BASE_RATE:,.0f}", delta_color="off")
        k4.metric("Expected hit rate", f"{expected / n_calls:.1%}",
                  f"random: {BASE_RATE:.1%}", delta_color="off")

        counts = (queue["priority_decile"].value_counts().reindex(range(1, 11), fill_value=0)
                  .rename("customers").rename_axis("priority decile"))
        st.bar_chart(counts, x_label="Priority decile (1 = call first)", y_label="Customers")

        show = (["rank"] + [c for c in queue.columns if c.lower() in ("customer_id", "id")] +
                ["recommendation", "priority_decile", "expected_rate", "score"] + features.RAW_COLUMNS)
        st.dataframe(as_percent(top[show].round(3), ["expected_rate"]), hide_index=True)
        st.download_button("Download the full queue (CSV)", queue[show].to_csv(index=False).encode(),
                           file_name="calling_queue.csv", mime="text/csv")

        with st.expander("Demo only: check the queue against what really happened"):
            st.caption("For a demonstration with test-set customers, whose real outcome is known. "
                       "The answer key must have customer_id and actually_subscribed columns.")
            key_file = st.file_uploader("Answer key (CSV)", type=["csv"], key="answer_key")
            if os.path.exists(DEMO_ANSWERS) and st.button("Use the demo answer key"):
                st.session_state["use_demo_answers"] = True
            answers = None
            if key_file is not None:
                answers = pd.read_csv(key_file)
            elif st.session_state.get("use_demo_answers"):
                answers = pd.read_csv(DEMO_ANSWERS)
            if answers is not None and "customer_id" in queue.columns:
                checked = queue.merge(answers[["customer_id", "actually_subscribed"]], on="customer_id")
                hits = int(checked.head(n_calls)["actually_subscribed"].sum())
                total = int(checked["actually_subscribed"].sum())
                a1, a2, a3 = st.columns(3)
                a1.metric(f"Real subscribers in top {n_calls}", f"{hits}",
                          f"expected {expected:.0f}", delta_color="off")
                a2.metric("Real hit rate", f"{hits / n_calls:.1%}",
                          f"random: {checked['actually_subscribed'].mean():.1%}", delta_color="off")
                a3.metric("Share of all subscribers reached", f"{hits / max(total, 1):.0%}",
                          f"with {n_calls / len(checked):.0%} of calls", delta_color="off")
                by_decile = (checked.groupby("priority_decile")
                             .agg(customers=("rank", "size"), real_rate=("actually_subscribed", "mean"),
                                  expected_rate=("expected_rate", "first")))
                st.dataframe(as_percent(by_decile, ["real_rate", "expected_rate"]))
            elif answers is not None:
                st.error("The uploaded customers have no customer_id column to match on.")

# ------------------------------------------------------------------ tab 3: how it works
with tab3:
    st.subheader("How every decision is made")
    st.markdown(f"""
1. **Prepare** the 15 raw columns with Member 2's feature rules and fitted pipeline
   (scaling, one-hot encoding and the same 30 selected columns). Nothing is refitted.
2. **Score** with Member 3's tuned **{bundle['model_name']}**: the "yes" column of `predict_proba`.
3. **Decide**: score at or above **{THRESHOLD:.3f}** means **CALL**. The threshold maximised F1
   on out-of-fold training predictions (Part 2).
4. **Prioritise**: the score is placed into one of 10 deciles using 9 cut points fixed in Part 2,
   and shown with that decile's real historical subscription rate.
""")
    table = features.decile_ranges(bundle["decile_edges"])
    table["expected rate (training)"] = table["decile"].map(RATES)
    if bundle.get("decile_table") is not None:
        test = bundle["decile_table"]
        table["real rate (test set)"] = table["decile"].map(test["conversion_rate"])
        table["lift vs random"] = table["decile"].map(test["lift"])
    if "lift vs random" in table.columns:
        table["lift vs random"] = table["lift vs random"].round(2)
    st.dataframe(as_percent(table, [c for c in table.columns if "rate" in c]), hide_index=True)
    st.caption("Lift = decile rate / overall rate. Lift 3 means three times better than calling at random.")
'''

README_MD = r'''# Term Deposit Call Planner: implementation guide

Member 4 (IT24100001), IT3091 Bank Marketing project.

This folder is the working app. It scores bank customers with **Member 3's tuned model**, applies the **decision rules from Part 2** (call threshold and priority deciles), and shows the result in a web page with two lenses:

| Tab | Lens | What it does |
|---|---|---|
| Single customer | ML lens | Enter one customer, get CALL or not, a priority decile and an expected subscription rate |
| Campaign queue | Analytical lens | Upload a CSV of customers, get a ranked calling queue and the effect of a call budget |
| How it works | Both | The rules behind every decision, for the presentation |

Nothing is trained in the app. It only loads saved objects.

---

## 1. What is in this folder

| File | What it is | Created by |
|---|---|---|
| `app.py` | The web app (Streamlit) | Part 3 notebook |
| `features.py` | Member 2's feature rules and the scoring helpers. **Must stay next to `app.py`** | Part 3 notebook |
| `best_pipeline.joblib` | Member 2's fitted feature pipeline + Member 3's model + Part 2 rules, in one file | Part 3 notebook |
| `requirements.txt` | Exact library versions used in Colab | Part 3 notebook |
| `demo_new_customers.csv` | 200 real test-set customers, outcome removed (never seen by any model) | Part 3 notebook |
| `demo_answer_key.csv` | What those 200 customers really did, to reveal after the demo | Part 3 notebook |
| `README.md` | This guide | Part 3 notebook |

---

## 2. Before you start

You need these in your `IT24100001` Drive folder, from earlier parts:

- `IT24100001_part2_threshold_deciles.joblib`, saved by the last cell of your Part 2

Plus the Member 1, 2 and 3 files that Part 2 already used.

All notebooks expect this layout in **your My Drive**:

```
My Drive/
└── ML & DWBI Assignment/
    └── ML Assignment/
        ├── IT24102060/   (Member 1)
        ├── IT24101461/   (Member 2)
        ├── IT24102228/   (Member 3)
        └── IT24100001/   (you: Part 1, 2, 3 outputs and the app folder)
```

---

## 3. Step 1: Run the Part 3 notebook in Colab

1. Upload `IT24100001_Part3_Export_and_App.ipynb` to Colab.
2. **Runtime → Run all.** Allow Google Drive access when asked.
3. Check these outputs as it runs:

| Section | What you must see |
|---|---|
| 1 | Every required file listed as `OK` or `OK (copy)` |
| 3 | `PASS` for: features.py reproduces Member 2, chained pipeline equals Member 3's model, deciles match Part 2 |
| 6 | `best_pipeline.joblib` saved, and the reload test says `PASS` |
| 7 | The `IT24100001/app` folder listing shows all 7 files |
| 9 | `CONFIRMED: nothing was written to the Member 1, 2 or 3 folders` |

If a check fails, nothing is saved and the message says why.

4. Note the **Python version** printed at the top of `requirements.txt` (section 7 shows it). You need the same version on your laptop.

---

## 4. Step 2: Download the app folder

1. In Google Drive, open `ML & DWBI Assignment / ML Assignment / IT24100001`.
2. Right-click the **`app`** folder → **Download**. Drive gives you a `.zip`.
3. Unzip it somewhere simple, for example `Documents/call_planner`.

Check that all 7 files from section 1 are inside the same folder.

---

## 5. Step 3: Install Python on your laptop

1. Check whether Python is installed. Open a terminal (Windows: **Command Prompt**; macOS: **Terminal**) and type:
   ```
   python --version
   ```
   On macOS use `python3 --version`.
2. You need the **same major.minor version as Colab** (for example 3.12), shown at the top of `requirements.txt`. If it is missing or different, install it from python.org. On Windows, tick **"Add python.exe to PATH"** during installation.

Why the same version: `best_pipeline.joblib` is a saved Python object. Loading it with very different library versions can fail.

---

## 6. Step 4: Create a virtual environment and install the libraries

A virtual environment keeps these exact versions separate from anything else on your laptop.

**Windows (Command Prompt):**
```
cd C:\Users\<you>\Documents\call_planner
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```

**macOS / Linux (Terminal):**
```
cd ~/Documents/call_planner
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

When the environment is active, the prompt starts with `(.venv)`. Every time you open a new terminal, `cd` into the folder and run the `activate` line again before starting the app.

---

## 7. Step 5: Start the app

With the environment active, inside the folder:
```
streamlit run app.py
```

Your browser opens at **http://localhost:8501**. If it does not, open that address yourself.

To stop the app, go back to the terminal and press **Ctrl + C**.

The first time, Streamlit may ask for an email address. Just press Enter to skip.

---

## 8. Step 6: Demonstration script

### Tab 1: Single customer (ML lens)

1. In **Start from**, choose **"Real test customer: strongest prospect"**. These are real customers from the test set, which no model was trained on.
2. Press **Score this customer**. Point out:
   - **Recommendation: CALL**, because the score is above the tuned threshold.
   - **Priority decile 1 or 2**: call first.
   - **Expected subscription rate**: the share of similar training customers who really subscribed.
3. Now change some inputs by hand to make a weak prospect, for example: `contact` = unknown, `month` = may, `campaign` = 8, `housing` = yes, `poutcome` = unknown, `pdays` = -1. Press **Score this customer** again. The decile drops and the recommendation becomes **Lower-cost channel**.
4. Open **Why this result?** to show what drives the model.

What to say: *"The score comes from Member 3's tuned model. The call line and the priority groups come from our Part 2 rules, which were chosen on training data only."*

### Tab 2: Campaign queue (Analytical lens)

1. Press **Use the demo customers** (200 real test customers, outcome hidden).
2. Move the **"How many calls can the team make?"** slider. Show **expected subscribers** against **random calling**.
3. Show the decile bar chart and the ranked table. Press **Download the full queue** to show the deliverable a call centre would receive.
4. Open **"Demo only: check the queue against what really happened"** and press **Use the demo answer key**. This reveals how many of the top-ranked customers **really** subscribed, which proves the ranking works on unseen customers.

What to say: *"With the same number of calls, following the queue reaches far more subscribers than calling at random."*

### Tab 3: How it works

Walk through the four steps and the decile table: score range, expected rate from training, real rate on the test set, and lift.

---

## 9. Screenshots for the report

1. Tab 1 with a strong prospect (CALL, decile 1 or 2).
2. Tab 1 with a weak prospect (lower-cost channel, low decile).
3. Tab 2 with the demo customers loaded and the slider set (metrics and ranked table visible).
4. Tab 2 with the answer key revealed.
5. Tab 3 decile table.

Windows: **Win + Shift + S**. macOS: **Cmd + Shift + 4**.

---

## 10. Using your own new customer data

The CSV needs a header row with these 15 columns (any order; extra columns such as `customer_id` are kept in the output):

```
age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome
35,management,married,tertiary,no,1500,yes,no,cellular,15,may,2,-1,0,unknown
```

Comma or semicolon separated both work, so the original `bank-full.csv` format is accepted. `y` and `duration` are ignored if present.

Allowed values:

| Column | Values |
|---|---|
| `job` | admin., blue-collar, entrepreneur, housemaid, management, retired, self-employed, services, student, technician, unemployed, unknown |
| `marital` | married, single, divorced |
| `education` | primary, secondary, tertiary, unknown |
| `default`, `housing`, `loan` | yes, no |
| `contact` | cellular, telephone, unknown |
| `month` | jan, feb, mar, apr, may, jun, jul, aug, sep, oct, nov, dec |
| `poutcome` | success, failure, other, unknown |
| `pdays` | days since last contact in a previous campaign; **-1 means never contacted** |

Text is not case sensitive. A value never seen in training is accepted but carries no information, and the app shows a warning.

**Do not use `bank.csv` from the UCI website as "new" data.** It is a random 10% sample of `bank-full.csv`, so most of its rows were used for training. The demo file in this folder is the honest choice.

---

## 11. Troubleshooting

| Problem | Fix |
|---|---|
| `ModuleNotFoundError: No module named 'features'` | Start the app from inside the folder, and keep `features.py` next to `app.py` |
| `best_pipeline.joblib was not found` | Same as above: run `streamlit run app.py` from the folder that contains it |
| Errors mentioning `_RemainderColsList`, `force_int_remainder_cols` or unpickling | Library versions differ from Colab. Activate the virtual environment and run `pip install -r requirements.txt` again |
| Sidebar warning about the scikit-learn version | Same fix as above |
| `pip` cannot find a version in `requirements.txt` | Your Python version differs from Colab's. Install the version at the top of `requirements.txt` |
| macOS: `lightgbm` fails with `libomp` | Run `brew install libomp`, then reinstall |
| Windows PowerShell blocks `activate` | Use Command Prompt instead, or run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` |
| `streamlit` is not recognised | The virtual environment is not active. Run the `activate` line again |
| Port 8501 already in use | `streamlit run app.py --server.port 8502` |
| "Missing required columns" after upload | The CSV header must contain all 15 column names in section 10 |

---

## 12. How it works (for the viva)

For every customer, the app does four things. None of them learns from the new data.

1. **Prepare**: Member 2's 8 feature rules, then Member 2's fitted pipeline (`transform` only: scaling with training averages, one-hot encoding with training categories, the same 30 selected columns).
2. **Score**: Member 3's tuned model, `predict_proba`, "yes" column.
3. **Decide**: score at or above the tuned threshold means **CALL**. The threshold maximised F1 on out-of-fold training predictions.
4. **Prioritise**: the score is compared with 9 fixed cut points to give a decile from 1 to 10, shown with that decile's real subscription rate from training.

The threshold, cut points and expected rates were fixed in Part 2, so the same customer always gets the same answer, whoever else is uploaded with them.
'''

# >>> MEMBER 4 FOLDER: the app itself and its guide
write_text("app.py", APP_PY, "Streamlit app")
write_text("README.md", README_MD, "implementation guide")

[WRITE] Member 4 folder -> IT24100001/app/app.py   (Streamlit app)
[WRITE] Member 4 folder -> IT24100001/app/README.md   (implementation guide)


'/content/drive/MyDrive/ML & DWBI Assignment/ML Assignment/IT24100001/app/README.md'

In [11]:
# >>> MEMBER 4 FOLDER: pinned requirements (same versions as this Colab session)
model_lib = {"LightGBM": "lightgbm", "XGBoost": "xgboost"}.get(FINAL_MODEL)
req = [f"# Created by Part 3 in Colab. Use Python {versions['python'].rsplit('.', 1)[0]}.x on your laptop.",
       f"scikit-learn=={versions['scikit-learn']}",
       f"numpy=={versions['numpy']}",
       f"pandas=={versions['pandas']}",
       f"joblib=={versions['joblib']}"]
if model_lib:
    req.append(f"{model_lib}=={versions[model_lib]}")
req.append("streamlit>=1.40")
write_text("requirements.txt", "\n".join(req) + "\n", "pinned library versions")
print("\n".join(req))

[WRITE] Member 4 folder -> IT24100001/app/requirements.txt   (pinned library versions)
# Created by Part 3 in Colab. Use Python 3.13.x on your laptop.
scikit-learn==1.6.1
numpy==2.1.3
pandas==2.2.3
joblib==1.6.0
xgboost==3.4.1
streamlit>=1.40


In [12]:
# >>> MEMBER 4 FOLDER: demo data = real test customers the models never saw
demo_idx = np.random.RandomState(RANDOM_STATE).choice(len(test_raw), size=200, replace=False)
demo = test_raw.iloc[demo_idx][RAW].reset_index(drop=True)
demo.insert(0, "customer_id", [f"C{i:04d}" for i in range(1, len(demo) + 1)])
answers = pd.DataFrame({"customer_id": demo["customer_id"], "actually_subscribed": y_test[demo_idx]})

demo.to_csv(os.path.join(APP_DIR, "demo_new_customers.csv"), index=False)
log_write(os.path.join(APP_DIR, "demo_new_customers.csv"), "200 test customers, outcome hidden")
answers.to_csv(os.path.join(APP_DIR, "demo_answer_key.csv"), index=False)
log_write(os.path.join(APP_DIR, "demo_answer_key.csv"), "real outcomes for the demo customers")
print(f"Demo customers: {len(demo)}, of whom {answers.actually_subscribed.sum()} really subscribed")

print("\nApp folder now contains:")
for f in sorted(os.listdir(APP_DIR)):
    if not f.startswith((".", "__")):
        print(f"   {f:<28} {os.path.getsize(os.path.join(APP_DIR, f)) / 1e3:>9.1f} KB")

[WRITE] Member 4 folder -> IT24100001/app/demo_new_customers.csv   (200 test customers, outcome hidden)
[WRITE] Member 4 folder -> IT24100001/app/demo_answer_key.csv   (real outcomes for the demo customers)
Demo customers: 200, of whom 24 really subscribed

App folder now contains:
   README.md                         10.7 KB
   app.py                            14.9 KB
   best_pipeline.joblib             179.5 KB
   demo_answer_key.csv                1.6 KB
   demo_new_customers.csv            16.4 KB
   features.py                        5.4 KB
   requirements.txt                   0.2 KB


## 8. Preview: what the app will show for the demo customers
This calls `features.score_customers`, **the same function the app calls**, on the demo file loaded from disk. Then it peeks at the answer key, as you will in the demo.

In [13]:
demo_loaded = pd.read_csv(os.path.join(APP_DIR, "demo_new_customers.csv"))
queue = features.score_customers(demo_loaded, reloaded)
print("Top 10 of the calling queue:")
display(queue[["rank", "customer_id", "recommendation", "priority_decile", "expected_rate", "score",
               "age", "job", "month", "contact", "poutcome"]].head(10).round(3))

checked = queue.merge(answers, on="customer_id")
n = int(round(0.2 * len(checked)))
hits = int(checked.head(n)["actually_subscribed"].sum())
print(f"Calling the top 20% ({n} customers) reaches {hits} real subscribers "
      f"(hit rate {hits / n:.0%}); random calling would expect about {n * answers.actually_subscribed.mean():.0f}.")

Top 10 of the calling queue:


,rank,customer_id,recommendation,priority_decile,expected_rate,score,age,job,month,contact,poutcome
0,1,C0189,CALL,1,0.477,0.892,61,self-employed,jul,cellular,success
1,2,C0014,CALL,1,0.477,0.842,37,technician,feb,cellular,success
2,3,C0041,CALL,1,0.477,0.815,27,blue-collar,aug,cellular,success
3,4,C0007,CALL,1,0.477,0.802,31,management,mar,cellular,success
4,5,C0111,CALL,1,0.477,0.761,54,technician,oct,cellular,success
5,6,C0003,CALL,1,0.477,0.722,44,management,mar,cellular,success
6,7,C0009,CALL,1,0.477,0.540,23,student,nov,cellular,failure
7,8,C0027,CALL,1,0.477,0.515,46,services,apr,cellular,other
8,9,C0040,CALL,1,0.477,0.439,31,entrepreneur,nov,cellular,success
9,10,C0039,CALL,1,0.477,0.430,25,admin.,jun,cellular,unknown


Calling the top 20% (40 customers) reaches 15 real subscribers (hit rate 38%); random calling would expect about 5.


## 9. File access log
Confirms which member folders were read, and that every write went into your own folder.

In [14]:
LOG_FILE = os.path.join(MEMBER_4_DIR, "IT24100001_part3_file_access_log.csv")
log_write(LOG_FILE, "this access log")
log = pd.DataFrame(ACCESS_LOG)
log.to_csv(LOG_FILE, index=False)

print("Reads per member folder :", log[log.action == "READ"].groupby("folder used").size().to_dict())
print("Writes per member folder:", log[log.action == "WRITE"].groupby("folder used").size().to_dict())
assert set(log.loc[log.action == "WRITE", "folder used"]) == {"Member 4"}, "Something was written outside your folder!"
print("CONFIRMED: nothing was written to the Member 1, 2 or 3 folders.")
print(f"\nNEXT: download the folder {short(APP_DIR)} and follow README.md inside it.")
log

[WRITE] Member 4 folder -> IT24100001/IT24100001_part3_file_access_log.csv   (this access log)
Reads per member folder : {'Member 1': 1, 'Member 2': 2, 'Member 3': 1, 'Member 4': 2}
Writes per member folder: {'Member 4': 8}
CONFIRMED: nothing was written to the Member 1, 2 or 3 folders.

NEXT: download the folder IT24100001/app and follow README.md inside it.


,action,file owner,folder used,path,purpose
0,WRITE,Member 4,Member 4,IT24100001/app/features.py,Member 2 feature rules + scoring helpers
1,READ,Member 4,Member 4,IT24100001/IT24100001_part2_threshold_deciles....,Part 2 decision rules
2,READ,Member 1,Member 1,IT24102060/ds52-train_test_split.joblib,"raw columns, categories, demo customers"
3,READ,Member 2,Member 2,IT24101461/IT24101461_selected_data.joblib,processed test set for the checks
4,READ,Member 2,Member 2,IT24101461/IT24101461_feature_pipeline.joblib,fitted scaler + encoder + selector
5,READ,Member 3,Member 3,IT24102228/IT24102228_xgboost_tuned.joblib,final model: XGBoost
6,WRITE,Member 4,Member 4,IT24100001/app/best_pipeline.joblib,pipeline + Part 2 rules + app information
7,READ,Member 4,Member 4,IT24100001/app/best_pipeline.joblib,reload test
8,WRITE,Member 4,Member 4,IT24100001/app/app.py,Streamlit app
9,WRITE,Member 4,Member 4,IT24100001/app/README.md,implementation guide
